# TranAD Integration Guide
**Running TranAD on Your Custom Dataset**

TranAD is a Transformer-based deep learning model for multivariate time-series anomaly detection. It achieved state-of-the-art results on multiple benchmarks (VLDB 2022).

## 1. TranAD Architecture Overview

**Key Components:**
- **Transformer Encoder/Decoder**: Captures temporal dependencies better than LSTM
- **Self-Conditioning**: Uses prior timestep info during training to reduce distribution shift
- **Adversarial Training**: Generator + Discriminator for robust anomaly detection
- **Multi-Head Attention**: Focuses on different time windows simultaneously

**Why TranAD is Better:**
- ✓ Captures long-range dependencies (Transformer advantage)
- ✓ Resistant to distribution shift (self-conditioning)
- ✓ F1-score improvements: 8-15% over LSTM/OmniAnomaly
- ✓ 0.9934 AUC on SMD dataset vs 0.9925 for LSTM

## 2. Setup & Installation

In [ ]:
# Installation commands (run in terminal)
install_commands = """
# Step 1: Navigate to TranAD directory
cd d:\\NexAura\\TranAD

# Step 2: Install PyTorch (CPU or GPU)
# For GPU (recommended if available):
pip install torch==1.8.1 torchvision==0.9.1 torchaudio==0.8.1 -f https://download.pytorch.org/whl/torch_stable.html

# For CPU only:
pip install torch==1.8.1 torchvision==0.9.1 torchaudio==0.8.1

# Step 3: Install other requirements
pip install -r requirements.txt

# Verify installation
python -c "import torch; print('PyTorch:', torch.__version__)"
"""

print("TRANAD INSTALLATION STEPS")
print("="*70)
print(install_commands)
print("\n✓ After installation, you can run TranAD models")

## 3. Data Format & Structure

**Expected Input Shape:**
- **Train data**: `.npy` file with shape `(num_train_samples, num_features)`
- **Test data**: `.npy` file with shape `(num_test_samples, num_features)`
- **Labels** (optional): For evaluation, shape `(num_test_samples,)` with 0 (normal) or 1 (anomaly)

**Directory Structure:**
```
TranAD/data/your_dataset/
├── train/
│   └── your_dataset_train.npy     # (10000, 38) example
├── test/
│   └── your_dataset_test.npy      # (2000, 38) example
└── labels/
    └── your_dataset_test_labels.npy # (2000,) - optional
```

## 4. Convert CSV → TranAD Format

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import os

# Load your dataset
csv_path = '../models/OmniAnamoly/Production System Dataset.csv'
df = pd.read_csv(csv_path)

print("CONVERTING CSV TO TRANAD FORMAT")
print("="*60)
print(f"Original shape: {df.shape}")

# Data cleaning
df = df.dropna()
df = df.drop_duplicates()
numeric_df = df.select_dtypes(include=[np.number])

# Normalize data
scaler = StandardScaler()
data_normalized = scaler.fit_transform(numeric_df)

print(f"After cleaning: {data_normalized.shape}")

# Train/Test split (80/20)
split_idx = int(0.8 * len(data_normalized))
train_data = data_normalized[:split_idx]
test_data = data_normalized[split_idx:]

print(f"Train shape: {train_data.shape}")
print(f"Test shape:  {test_data.shape}")

# Create TranAD data directory
dataset_name = 'OmniAnamaly'  # Your dataset name
tranad_datadir = f'./data/{dataset_name}'
os.makedirs(f'{tranad_datadir}/train', exist_ok=True)
os.makedirs(f'{tranad_datadir}/test', exist_ok=True)

# Save as .npy files
np.save(f'{tranad_datadir}/train/{dataset_name}_train.npy', train_data.astype(np.float32))
np.save(f'{tranad_datadir}/test/{dataset_name}_test.npy', test_data.astype(np.float32))

print(f"\n✓ Data saved to: {tranad_datadir}")
print(f"  ├─ train/{dataset_name}_train.npy")
print(f"  └─ test/{dataset_name}_test.npy")

## 5. TranAD Model Architecture

**Key Components (in src/models.py):**
```python
class TranAD(nn.Module):
    # Transformer Encoder: Processes sequences
    - d_model: 128 (embedding dimension)
    - n_heads: 8 (multi-head attention)
    - num_layers: 1 (transformer blocks)
    - d_ff: 512 (feed-forward dim)
    
    # Self-Conditioning:
    - Uses encoder output z_t from prev training step
    - Reduces train/test distribution shift
    
    # Adversarial Training:
    - Generator: Creates reconstructions
    - Discriminator: Distinguishes normal vs reconstructed
```

**Configuration (src/params.json):**

In [ ]:
# Example configuration from params.json
config_example = {
    "OmniAnamaly": {
        "epochs": 5,
        "batch_size": 128,
        "slide_win": 12,              # Window size (timesteps)
        "dim": 38,                     # Number of features
        "d_model": 128,               # Transformer embedding dim
        "n_heads": 8,                 # Attention heads
        "n_layers": 1,                # Transformer layers
        "d_ff": 512,                  # Feed-forward dimension
        "dropout": 0.0,
        "lr": 0.001,                 # Learning rate
        "device": "cpu",              # or "cuda" for GPU
    }
}

print("PARAMETER CONFIGURATION")
print("="*60)
print("Key parameters for your dataset:")
for param, value in config_example["OmniAnamaly"].items():
    print(f"  {param:15} : {value}")

print("\n✓ Adjust these in src/params.json before training")

## 6. Running TranAD: Terminal Commands

In [ ]:
tranad_commands = """
# Navigate to TranAD directory
cd d:\\NexAura\\TranAD

================================
# TRAINING & TESTING
================================

# 1. Train & test TranAD on your dataset
python main.py --model TranAD --dataset OmniAnamaly --retrain

# 2. Train with 20% of data (for testing)
python main.py --model TranAD --dataset OmniAnamaly --retrain --less

# 3. Test with pre-trained model (no retraining)
python main.py --model TranAD --dataset OmniAnamaly

================================
# OTHER MODELS FOR COMPARISON
================================

# Try other models for comparison:
python main.py --model LSTM_AD --dataset OmniAnamaly --retrain
python main.py --model OmniAnomaly --dataset OmniAnamaly --retrain
python main.py --model USAD --dataset OmniAnamaly --retrain
python main.py --model GDN --dataset OmniAnamaly --retrain

================================
# OUTPUT
================================
# Results will be printed including:
# - Training loss over epochs
# - Testing metrics: Precision, Recall, F1, AUC
# - Threshold for anomaly detection
# - Results saved in: results/
"""

print("TRANAD EXECUTION COMMANDS")
print("="*70)
print(tranad_commands)

## 7. Expected Output & Metrics

In [ ]:
# Example output from main.py
example_output = """
Creating new model: TranAD
Training TranAD on OmniAnamaly
Epoch 0,        L1 = 0.098
Epoch 1,        L1 = 0.039
Epoch 2,        L1 = 0.022
Epoch 3,        L1 = 0.018
Epoch 4,        L1 = 0.016
100%|████████| 5/5 [00:03<00:00, 1.57it/s]
Training time:  3.19 s

Testing TranAD on OmniAnamaly
{
  'FN': 12,           # False Negatives
  'FP': 45,           # False Positives
  'TN': 7575,         # True Negatives
  'TP': 748,          # True Positives
  'f1': 0.891,        # F1-Score
  'precision': 0.804, # Precision
  'recall': 1.0,      # Recall
  'threshold': 0.161  # Anomaly detection threshold
}
"""

print("EXPECTED OUTPUT FROM TRANAD")
print("="*70)
print(example_output)

print("\nKEY METRICS:")
print("  • F1-Score: Balance between precision and recall (0-1, higher is better)")
print("  • Precision: Ratio of correctly detected anomalies")
print("  • Recall: Percentage of total anomalies found")
print("  • Threshold: Used to classify normal vs anomaly scores")

## 8. Post-Processing: Get Anomaly Scores

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load TranAD test data
test_data = np.load('./data/OmniAnamaly/test/OmniAnamaly_test.npy')

# Read results from main.py output
# The results are typically saved in results/ folder
# For now, simulate anomaly scores

print("POST-PROCESSING TRANAD RESULTS")
print("="*60)

# Generate anomaly scores (in practice, TranAD outputs these)
# Shape: (num_test_samples,)
anomaly_scores_tranad = np.random.random(len(test_data)) * 0.5
anomaly_scores_tranad[400:420] = np.random.random(20) * 1.5 + 0.5  # Anomalies

# Create results dataframe
threshold = 0.161  # From TranAD output
results_tranad = pd.DataFrame({
    'sample_index': np.arange(len(test_data)),
    'anomaly_score': anomaly_scores_tranad,
    'is_anomaly': anomaly_scores_tranad > threshold
})

print(f"Results shape: {results_tranad.shape}")
print(f"Total anomalies detected: {results_tranad['is_anomaly'].sum()}")
print(f"\nTop 10 anomalies by score:")
print(results_tranad.nlargest(10, 'anomaly_score')[['sample_index', 'anomaly_score']])

# Save results
results_tranad.to_csv('./tranad_results.csv', index=False)
print(f"\n✓ Results saved to: tranad_results.csv")

## 9. Visualization: Anomalies

In [ ]:
# Visualize TranAD results
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Plot 1: First feature over time with anomalies highlighted
feature_idx = 0
feature_data = test_data[:, feature_idx]

ax = axes[0]
ax.plot(feature_data, label='Test Data', color='blue', alpha=0.7)
anomaly_indices = results_tranad[results_tranad['is_anomaly']].index.values
ax.scatter(anomaly_indices, feature_data[anomaly_indices], 
          color='red', s=100, marker='x', label=f'Anomalies ({len(anomaly_indices)})', linewidth=2, zorder=5)
ax.set_title('TranAD: Feature Reconstruction with Detected Anomalies', fontsize=12, fontweight='bold')
ax.set_ylabel('Normalized Value')
ax.set_xlabel('Time Index')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Plot 2: Anomaly scores with threshold
ax = axes[1]
ax.plot(anomaly_scores_tranad, label='Anomaly Score', color='green', alpha=0.7, linewidth=1.5)
ax.axhline(y=threshold, color='red', linestyle='--', label=f'Threshold ({threshold:.3f})', linewidth=2)
ax.scatter(anomaly_indices, anomaly_scores_tranad[anomaly_indices], 
          color='red', s=100, marker='o', label='Detected Anomalies', zorder=5)
ax.set_title('TranAD: Anomaly Scores vs Threshold', fontsize=12, fontweight='bold')
ax.set_ylabel('Anomaly Score')
ax.set_xlabel('Time Index')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Visualization complete!")

## 10. Summary & Next Steps

In [ ]:
summary = """
╔════════════════════════════════════════════════════════════════════════════╗
║                     TRANAD QUICK REFERENCE GUIDE                          ║
╚════════════════════════════════════════════════════════════════════════════╝

1. DATA PREPARATION
   ✓ Convert CSV to .npy format (normalized)
   ✓ Split into train (80%) and test (20%)
   ✓ Place in TranAD/data/dataset_name/train/ and test/

2. TRAINING
   ✓ Update params.json with your dataset config
   ✓ Run: python main.py --model TranAD --dataset dataset_name --retrain
   ✓ Monitor training loss in terminal

3. EVALUATION
   ✓ Reports: F1-score, Precision, Recall, AUC
   ✓ Automatically determines optimal threshold
   ✓ Results printed on terminal

4. GET ANOMALY SCORES
   ✓ Extract from model output
   ✓ Compare with threshold
   ✓ Visualize results

5. ADVANTAGES OF TRANAD
   ✓ State-of-the-art (VLDB 2022)
   ✓ Better than LSTM/OmniAnomaly (+8-15% F1)
   ✓ Handles long-range dependencies with Transformers
   ✓ Self-conditioning reduces distribution shift
   ✓ Adversarial training improves robustness

6. COMPARISON TABLE (from paper)
   Dataset    | TranAD F1 | LSTM F1  | OmniAnomaly F1
   ────────────────────────────────────────────────
   SMD        | 0.9921    | 0.9788   | 0.9080
   SWaT       | 0.8915    | 0.8094   | 0.8265
   WADI       | 0.4951    | 0.3345   | 0.4260
   MSL        | 0.9916    | 0.9605   | 0.9292

7. TROUBLESHOOTING
   • CUDA out of memory? → Reduce batch_size in params.json
   • Slow training? → Use --less flag for 20% data
   • Bad results? → Try different slide_win (window size)
   • No GPU? → Set device: "cpu" in params.json (slower)

8. FILES STRUCTURE
   TranAD/
   ├── main.py              # Entry point
   ├── src/
   │   ├── models.py        # TranAD architecture
   │   ├── parameters.py    # hyperparameter loading
   │   └── utils.py         # Utility functions
   ├── data/
   │   └── dataset_name/
   │       ├── train/
   │       └── test/
   └── results/             # Output saved here
"""

print(summary)